In [26]:
import json
import requests
import pandas as pd
import kagglehub
from recognition_evaluators import evaluate_tm_df

In [27]:
BASE_URL = "https://deepcode.ci.nsu.ru"
api_key = "sk-d03991b7df2a4bd0b306fe9dbb2d6389"
headers = {"Authorization": f"Bearer {api_key}"}

MODEL_ID = "Qwen3.8-27B"
print("Model:", MODEL_ID)


Model: Qwen3.8-27B


In [28]:
multiclass_dataset_path = kagglehub.dataset_download("sagarikashreevastava/cognitive-distortion-detetction-dataset")
multiclass_dataset_file_path = multiclass_dataset_path + "/Annotated_data.csv"

df = pd.read_csv(multiclass_dataset_file_path)
df = df.drop("Id_Number", axis=1)

held_fraction = 5
train_df = df[df.index % held_fraction != 0].reset_index(drop=True)
held_df = df[df.index % held_fraction == 0].reset_index(drop=True)

df

,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional)
0,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,Personalization,NaN
1,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,Labeling,Emotional Reasoning
2,So I’ve been dating on and off this guy for a...,NaN,No Distortion,NaN
3,My parents got divorced in 2004. My mother has...,NaN,No Distortion,NaN
4,I don’t really know how to explain the situati...,I refused to go because I didn’t know if it wa...,Fortune-telling,Emotional Reasoning
...,...,...,...,...
2525,I’m a 21 year old female. I spent most of my l...,NaN,No Distortion,NaN
2526,I am 21 female and have not had any friends fo...,Now I am at university my peers around me all ...,Overgeneralization,NaN
2527,From the U.S.: My brother is 19 years old and ...,He claims he’s severely depressed and has outb...,Mental filter,Mind Reading
2528,From the U.S.: I am a 21 year old woman who ha...,NaN,No Distortion,NaN


In [29]:
def choose_text(row):
    if pd.notna(row["Distorted part"]):
        return row["Distorted part"]
    return row["Patient Question"]

def collect_labels(row):
    labels = []
    dominant = row["Dominant Distortion"]
    if dominant != "No Distortion":
        labels.append(dominant)
    secondary = row["Secondary Distortion (Optional)"]
    if pd.notna(secondary):
        if secondary != "No Distortion" and secondary not in labels:
            labels.append(secondary)
    return labels

train_data = pd.DataFrame()
train_data["text"] = train_df.apply(choose_text, axis=1)
train_data["labels"] = train_df.apply(collect_labels, axis=1)
train_data["dominant_label"] = train_df["Dominant Distortion"]

held_data = pd.DataFrame()
held_data["text"] = held_df.apply(choose_text, axis=1)
held_data["labels"] = held_df.apply(collect_labels, axis=1)

all_labels = [
    "All-or-nothing thinking",
    "Emotional Reasoning",
    "Fortune-telling",
    "Labeling",
    "Magnification",
    "Mental filter",
    "Mind Reading",
    "Overgeneralization",
    "Personalization",
    "Should statements"]

train_data

,text,labels,dominant_label
0,I feel trapped inside my disgusting self and l...,"[Labeling, Emotional Reasoning]",Labeling
1,So I’ve been dating on and off this guy for a...,[],No Distortion
2,My parents got divorced in 2004. My mother has...,[],No Distortion
3,I refused to go because I didn’t know if it wa...,"[Fortune-telling, Emotional Reasoning]",Fortune-telling
4,"About a year ago to the month, I was in the mi...",[],No Distortion
...,...,...,...
2019,From the U.S.: I’m a 12th grader in high schoo...,[],No Distortion
2020,Now I am at university my peers around me all ...,[Overgeneralization],Overgeneralization
2021,He claims he’s severely depressed and has outb...,"[Mental filter, Mind Reading]",Mental filter
2022,From the U.S.: I am a 21 year old woman who ha...,[],No Distortion


In [30]:
def build_balanced_train(train_data):
    label_order = sorted(train_data["dominant_label"].unique())
    label_groups = {}
    label_positions = {}

    for label in label_order:
        label_groups[label] = train_data[train_data["dominant_label"] == label].sample(frac=1, random_state=42).reset_index(drop=True)
        label_positions[label] = 0

    balanced_rows = []

    while len(balanced_rows) < len(train_data):
        for label in label_order:
            if label_positions[label] < len(label_groups[label]):
                balanced_rows.append(label_groups[label].iloc[label_positions[label]])
                label_positions[label] += 1

    return pd.DataFrame(balanced_rows).reset_index(drop=True)

balanced_train = build_balanced_train(train_data)

balanced_train.head(15)

,text,labels,dominant_label
0,"Then, she turned the same volatile screaming o...",[All-or-nothing thinking],All-or-nothing thinking
1,ive had a deep seeded hate and anger with me f...,[Emotional Reasoning],Emotional Reasoning
2,I feel I am useless to my kids or my husband b...,[Fortune-telling],Fortune-telling
3,I also tried to cut myself three times in my w...,[Labeling],Labeling
4,"I’m lost in life, I often feel rage, anger and...","[Magnification, Labeling]",Magnification
5,Everyday I feel my life is being wasted.,"[Mental filter, Magnification]",Mental filter
6,They seem not to love and support me.,[Mind Reading],Mind Reading
7,I am seventeen and I dated this guy for about ...,[],No Distortion
8,I also am very unorganized and messy and I str...,[Overgeneralization],Overgeneralization
9,Sometimes i get awful thoughts that makes me f...,"[Personalization, Mind Reading]",Personalization


In [31]:
def labels_to_json(labels):
    return json.dumps(labels, ensure_ascii=False)

def build_many_shot_prefix(train_df, allowed_labels):
    parts = []
    parts.append(
        "You are a professional psychotherapist experienced in cognitive-behavioral therapy.\n"
        "Your task is multi-label classification of cognitive distortions in patient texts.\n"
        "You can assign none, one, or several cognitive distortion labels to each text.\n"
        "Allowed labels are:")

    parts.append(labels_to_json(allowed_labels))

    parts.append(
        "Important annotation rule: if there is no cognitive distortion, return an empty JSON array: [].\n"
        "Do not use the label 'No Distortion'. Absence of distortion must be represented only as [].\n"
        "Return only a valid JSON array of strings. Do not return explanations, comments, markdown, or extra text.\n"
        "Below are annotated examples.")

    for i, row in train_df.iterrows():
        parts.append(f"\nExample {i + 1}:")
        parts.append(f"Text: {row['text']}")
        parts.append(f"Labels: {labels_to_json(row['labels'])}")

    parts.append("\nNow classify new texts using the same label set and the same annotation logic.")
    return "\n".join(parts)

def build_test_prompt(many_shot_prefix, text):
    return many_shot_prefix + "\n\nText to classify:\n" + text + "\n\nReturn only the JSON array of labels:"

In [32]:
selected_examples = balanced_train
many_shot_prefix = build_many_shot_prefix(selected_examples, all_labels)
print("Training examples:", len(selected_examples))
print("Held-out rows:", len(held_data))

Training examples: 2024
Held-out rows: 506


In [33]:
MAX_OUTPUT_TOKENS = 512

response_schema = {
    "type": "array",
    "items": {
        "type": "string",
        "enum": all_labels}}

def call_qwen(prompt):
    global last_response
    response = requests.post(
        BASE_URL + "/api/chat/completions",
        headers=headers,
        json={
            "model": MODEL_ID,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "chat_template_kwargs": {"enable_thinking": False},
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "cognitive_distortions",
                    "schema": response_schema}},
            "max_tokens": MAX_OUTPUT_TOKENS,
            "stream": False
        },
        timeout=(300, 3600)
    )
    response.raise_for_status()
    last_response = response.json()
    return last_response["choices"][0]["message"]["content"]

def parse_model_answer(raw_answer):
    return json.loads(raw_answer)

In [34]:
results = []

In [35]:
for _, row in held_data.iloc[len(results):].iterrows():
    text = row["text"]
    raw_answer = call_qwen(build_test_prompt(many_shot_prefix, text))
    results.append({
        "text": text,
        "true_labels": row["labels"],
        "predicted_labels": parse_model_answer(raw_answer)})

results_df = pd.DataFrame(results)
results_df

,text,true_labels,predicted_labels
0,The voice are always fimilar (someone she know...,[Personalization],[Fortune-telling]
1,Hello. I have been friend with a guy since gra...,[],[Mind Reading]
2,"I thought that he displayed traits of honor, l...",[Labeling],[Labeling]
3,"I started going to therapy in December, after ...",[],[Personalization]
4,During this time I was recruited to many great...,[Fortune-telling],[Fortune-telling]
...,...,...,...
501,"Hi there, my mother was diagnosed with Bipolar...",[],[]
502,"Lately, I’ve been feeling like someone is watc...",[Emotional Reasoning],"[Fortune-telling, Emotional Reasoning]"
503,I have had anxiety almost all of my life but l...,[Mental filter],[Overgeneralization]
504,"Hi, for about 3 years now I have been feeling ...",[],"[Fortune-telling, Mind Reading]"


In [36]:
metric_names = [
    "All-or-nothing_thinking",
    "Fortune-telling",
    "Mental_filter",
    "Overgeneralization",
    "Labeling",
    "Mind_Reading",
    "Personalization",
    "Emotional_Reasoning",
    "Magnification",
    "Should_statements"]

def llm_evaluator(all_metrics, predictions, text, threshold, min_count):
    predicted_labels = next(predictions)
    predicted_labels = [label.replace(" ", "_") for label in predicted_labels]
    distortions_by_metric = {}

    for metric in all_metrics:
        distortions_by_metric[metric] = metric in predicted_labels

    return distortions_by_metric

predictions = iter(results_df["predicted_labels"])
precision, recall, f1, accuracy = evaluate_tm_df(held_df, predictions, llm_evaluator, 0, metric_names)

metrics_df = pd.DataFrame({
    "metric": metric_names,
    "precision": [precision[metric] for metric in metric_names],
    "recall": [recall[metric] for metric in metric_names],
    "f1": [f1[metric] for metric in metric_names],
    "accuracy": [accuracy[metric] for metric in metric_names]})

macro_f1 = sum(f1.values()) / len(metric_names)
summary_df = pd.DataFrame([{
    "model": MODEL_ID,
    "training_examples": len(selected_examples),
    "test_rows": len(held_data),
    "macro_f1": macro_f1}])

summary_df

,model,training_examples,test_rows,macro_f1
0,Qwen3.8-27B,2024,506,0.430911


In [37]:
metrics_df

,metric,precision,recall,f1,accuracy
0,All-or-nothing_thinking,0.391304,0.360000,0.375000,0.940711
1,Fortune-telling,0.433735,0.750000,0.549618,0.883399
2,Mental_filter,0.175439,0.333333,0.229885,0.867589
3,Overgeneralization,0.313253,0.440678,0.366197,0.822134
4,Labeling,0.508197,0.756098,0.607843,0.920949
5,Mind_Reading,0.400000,0.723404,0.515152,0.873518
6,Personalization,0.392857,0.500000,0.440000,0.889328
7,Emotional_Reasoning,0.281250,0.257143,0.268657,0.903162
8,Magnification,0.583333,0.396226,0.471910,0.907115
9,Should_statements,0.410256,0.592593,0.484848,0.932806
